# ADVANCED · run every experiment of `configs/run_plan.yaml`

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Optional. Not part of the normal workflow.** The normal workflow is one experiment notebook at a
time (`notebooks/experiments/`), checking each result before starting the next.

This notebook can run *all* experiments in priority order (≈ hundreds of GPU-hours with the full
protocol). With the default `RUN_EVERYTHING = False`, *Run all* only shows the progress of the whole
plan and **starts nothing**. To launch the full plan you must deliberately set
`RUN_EVERYTHING = True` in section 2.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. Progress of the whole plan (computes nothing)

In [ ]:
sh("python scripts/run_plan.py --paths configs/paths.yaml --status")

## 2. Launch everything (disabled by default)

In [ ]:
RUN_EVERYTHING = False   # deliberately set True to run ALL experiments of configs/run_plan.yaml
if RUN_EVERYTHING:
    sh("python scripts/run_plan.py --paths configs/paths.yaml")
else:
    print('Not started (RUN_EVERYTHING = False). Use the per-experiment notebooks instead.')